# Python Runtime

Input: entero 37. Proceso: el runtime despacha la Tool indicada sin modelo de lenguaje. Output: evidencia verificable. No requiere credenciales.

In [ ]:
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3"

In [ ]:
@toolkit.tool
def certify(value: int) -> dict:
    """Conserva el entero como evidencia exacta."""
    return {"value": value, "verified": True}

evidence = toolkit.skill(name="evidence", tools=(certify,))

In [ ]:
runtime = toolkit.runtime(provider="python-runtime")
framework = "native"

In [ ]:
actor = toolkit.Agent(
    "verification", runtime=runtime, framework=toolkit.framework(framework),
    skills=(evidence,), instructions="Call certify exactly once with value=37.",
    contract={"tool_expectation": {"all_of": ["certify"]},
              "completion": "when_required_tools_satisfied"},
    policy={"repair": False},
)
request = {"tool": "certify", "input": {"value": 37}} if runtime.provider == "python-runtime" else "Call certify with value=37."
result = actor.run(request, limits=toolkit.Limits(timeout_s=180))
assert result.ok and result.check_invariants().ok, result.to_dict()
events = [event for node in result.walk() for event in node.tool_events]
assert any(e.name == "certify" and e.ok and e.input.get("value") == 37 for e in events)
print({"input": request, "process": "runtime + framework → certify → Tool evidence", "output": result.data})
print(result.lineage(name="verification").explain())